In [1]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

client = AsyncOpenAI()
llm = llm_factory("gpt-5-mini", client=client)
scorer = Faithfulness(llm=llm)

/Users/satvik/Desktop/Project/personal/RAGAS/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Example 1: Response is mostly grounded but adds one unsupported claim
# 'has been proven to prevent all forms of cancer' is not in the context
result = await scorer.ascore(
    user_input="What are the health benefits of green tea?",
    response="Green tea contains antioxidants that help reduce inflammation. It also boosts metabolism and has been proven to prevent all forms of cancer.",
    retrieved_contexts=[
        "Green tea is rich in antioxidants, particularly catechins, which help reduce inflammation and oxidative stress.",
        "Studies suggest green tea may modestly boost metabolic rate."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 0.5


In [3]:
# Example 2 : Every claim in the response is directly supported by the context
result = await scorer.ascore(
    user_input="When was the first Super Bowl played?",
    response="The first Super Bowl was played on January 15, 1967, at the Los Angeles Memorial Coliseum.",
    retrieved_contexts=[
        "The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles, California."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 1.0


In [4]:
# Example 3: Response introduces multiple facts not found in the context
# Context only states the speed; travel time to Earth and Earth-circling claim are hallucinated
result = await scorer.ascore(
    user_input="What is the speed of light?",
    response="The speed of light is approximately 3x10^8 meters per second. It takes light about 8 minutes to travel from the Sun to Earth, and light can circle the Earth 7.5 times in one second.",
    retrieved_contexts=[
        "The speed of light in a vacuum is approximately 299,792,458 meters per second."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 0.3333333333333333
